# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Set OpenAI key in the environment
from google.colab import userdata
import os

api_key = userdata.get('OPENAI_API_KEY_RAG')
os.environ['OPENAI_API_KEY'] = api_key

In [3]:
from openai import OpenAI
from IPython.display import Markdown

client = OpenAI()

In [4]:
%cd /content/drive/MyDrive/RAG Bootcamp/RAG/RAG with OpenAI File Search

/content/drive/MyDrive/RAG Bootcamp/RAG/RAG with OpenAI File Search


# Create the Knowledge Component

In [5]:
# Define the place where the docs are and extensions
docs_directory = "Docs"
supported_extensions = [".txt", ".py"]

In [10]:
# Append all file names and paths together
files_path = []

for filename in os.listdir(docs_directory):
  file_path = os.path.join(docs_directory, filename)
  if os.path.isfile(file_path):
      file_extension = os.path.splitext(filename)[1].lower()
      if file_extension in supported_extensions:
          files_path.append(file_path)


print(f"the files are {files_path}")

the files are ['Docs/7.02 Introduction to Multimodal RAG.txt', 'Docs/7.01 Game Plan for Multimodal RAG.txt', 'Docs/7.11 Python - Tokenize text.txt', 'Docs/7.04 Python - Extracting Audio.txt', 'Docs/7.09 Introduction to Contrastive Learning.txt', 'Docs/7.13 Python - Embedding Images.txt', 'Docs/7.17 Python - Query Embedding.txt', 'Docs/7.12 Python - Chunking and Embedding Text.txt', 'Docs/7.03 Python - Sep up and Video.txt', 'Docs/7.07 Python - Transcribing Audio with OpenAI Whisper.txt', 'Docs/7.07 Whisper Model.txt', 'Docs/7.08 Python - Extracting Frames.txt', 'Docs/7.05 Python - Compressing Audio.txt', 'Docs/7.10 Understanding the CLIP Model.txt', 'Docs/7.14 Understanding Cosine Similarity in Multimodal Contexts.txt', 'Docs/7.20 Python - GenAI Model Setup.txt', 'Docs/7.22 What did we learn in this section.txt', 'Docs/multimodal_rag.py', 'Docs/7.21 Python - GenAI model.txt', 'Docs/7.18 Python - Cosine Similarity of Query and Text.txt', 'Docs/7.16 Python - Visualizing Text and Images.t

In [13]:
# create vector store
vector_store = client.vector_stores.create(
    name="RAG-Bootcamp Vector Store",
)

In [14]:
# Retrieve the id
print(vector_store.id)

vs_6af0aca4fecc81918ca1e12df5d81c86


In [15]:
# Upload filees to OpenAI
file_ids = []
for file_path in files_path:
    with open(file_path, "rb") as file:
        response = client.files.create(
            file=file,
            purpose='assistants'
        )
        file_ids.append(response.id)

In [16]:
# add files to the vector store
for file_id in file_ids:
    result = client.vector_stores.files.create(
        vector_store_id=vector_store.id,
        file_id=file_id
    )

# Rag system

In [25]:
# call the API
response = client.responses.create(
    model = 'gpt-5.2',
    input = "When should I build multimodal RAG",
    tools = [
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id],
            "max_num_results": 5
        },
    ],
    include = ["file_search_call.results"],
    instructions = "Only answer based on the information retrieved. Say you dont know if you dont know"
)

In [26]:
# Explore the output
display(Markdown(response.output_text))

Build multimodal RAG when the knowledge you need to retrieve is **not fully captured in text**, and you need to **pull relevant evidence from multiple media types (images, audio, video) and then generate a text output**. Multimodal RAG is specifically described as retrieving and generating across data types to “significantly enhance” understanding compared to using isolated sources .

Concretely, you should build multimodal RAG if:

- **Your key signals live across modalities**, e.g. product spec docs (text) + competitor photos (images) + focus-group recordings (audio) + customer feedback clips (video). The example given is a marketing team that needs to quickly synthesize insights from all these formats into one coherent output .
- **You can’t realistically review everything manually**, and you need the system to retrieve “the most relevant insights” from things like videos and photos, then combine them with text to produce a single report .
- **“Understanding” multiple modalities is the goal** (inputs are strong for text/image/audio; video understanding is improving, with a practical workaround of using extracted frames + text) .
- **You’re prepared for added complexity and cost**, because each modality needs different processing (tokenization/embeddings for text; feature extraction for images; transforming audio; video processing frame-by-frame) and then synchronization + weighing relevance across modalities .

A practical sign you’re ready: you have (or can build) the pipeline to **turn video into frames + audio transcripts**, align image/text using models like **CLIP**, and then retrieve the best-matching chunks/frames before generation—exactly the kind of workflow outlined for multimodal RAG systems .

In [27]:
# Explore the output
response.output[0].queries

['When should I build multimodal RAG?',
 'multimodal RAG when to use',
 'criteria decide multimodal retrieval augmented generation images audio video',
 'use cases for multimodal RAG vs text-only RAG']

In [28]:
# Explore teh results
response.output[0].results[0]

Result(attributes={}, file_id='file-TryBUxW68KmxdSG2NTdw9H', filename='7.02 Introduction to Multimodal RAG.txt', score=0.4785, text="Here's how it works. So you provide AI with a text, with document features, or with product features, a series of compact images showcasing their latest designs, customer feedback videos, audio recordings. And then the model processes each type of data through specialized neural networks. So text is handled by natural language processing techniques, images go through convolutional neural networks, audio is analyzed using spectrograms, and video is, you know, frame by frame in order to understand the visual context. And then after processing each modality, the AI synthesizes the information to create a cohesive marketing strategy. So it identifies key selling points from the text, spots design trends from competitor images, captures emotional tones from audio clips, and summarizes customer sentiments from videos. Then the challenge is how do we integrate a